# 02 · PySpark Transformations & Analytics

This notebook uses a small retail sales dataset to practice with PySpark DataFrames and then reproduce part of the analysis in SQL.

### Skills demonstrated
- DataFrame creation and typing
- Column expressions and derived metrics
- Filtering and date conditions
- groupBy and aggregations
- DataFrame joins
- Window functions
- Temporary views and SQL equivalents


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## 1. Build the sales and region datasets



In [ ]:
raw_sales = spark.createDataFrame(
    [
        ("2024-01-01", "S001", "Electronics", "Laptop", 999.99, 2, "West"),
        ("2024-01-01", "S002", "Books", "Python Guide", 49.99, 5, "East"),
        ("2024-01-02", "S003", "Electronics", "Phone", 699.99, 3, "West"),
        ("2024-01-02", "S004", "Clothing", "Jacket", 129.99, 4, "North"),
        ("2024-01-03", "S005", "Books", "Data Science", 59.99, 8, "East"),
        ("2024-01-03", "S006", "Electronics", "Tablet", 449.99, 2, "South"),
        ("2024-01-04", "S007", "Clothing", "Shoes", 89.99, 6, "West"),
        ("2024-01-04", "S008", "Electronics", "Earbuds", 79.99, 10, "North"),
        ("2024-01-05", "S009", "Books", "ML Handbook", 69.99, 3, "South"),
        ("2024-01-05", "S010", "Clothing", "Hat", 29.99, 15, "East"),
    ],
    ["date", "sale_id", "category", "product", "price", "quantity", "region"],
)

sales = raw_sales.withColumn("date", F.to_date("date"))

regions = spark.createDataFrame(
    [
        ("West", "Pacific", "Sarah"),
        ("East", "Atlantic", "Mike"),
        ("North", "Central", "Lisa"),
        ("South", "Gulf", "Tom"),
    ],
    ["region", "territory", "manager"],
)

print(f"Sales rows: {sales.count()}")
print(f"Region rows: {regions.count()}")
display(sales)


## 2. Create derived columns

Revenue is calculated as price × quantity, while discounted_price applies a 10% discount to the unit price.


In [ ]:
revenue_view = sales.select(
    "sale_id",
    "product",
    "category",
    "price",
    "quantity",
    F.round(F.col("price") * F.col("quantity"), 2).alias("total_revenue"),
    F.round(F.col("price") * 0.90, 2).alias("discounted_price"),
)

display(revenue_view)


## 3. Filter records


In [ ]:
# High-value unit prices
high_price_sales = sales.filter(F.col("price") > 100)
display(high_price_sales)

# Electronics sold in the West
west_electronics = sales.filter(
    (F.col("category") == "Electronics") & (F.col("region") == "West")
)
display(west_electronics)

# Inclusive date range
jan_2_to_4 = sales.filter(F.col("date").between("2024-01-02", "2024-01-04"))
display(jan_2_to_4)


## 4. Aggregate sales performance

The next transformations summarize revenue by category and operating metrics by region.


In [ ]:
category_summary = (
    sales.groupBy("category")
    .agg(
        F.count("sale_id").alias("num_sales"),
        F.round(F.sum(F.col("price") * F.col("quantity")), 2).alias("total_revenue"),
    )
    .orderBy(F.col("total_revenue").desc())
)

display(category_summary)


In [ ]:
region_summary = (
    sales.groupBy("region")
    .agg(
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.sum("quantity").alias("total_quantity"),
    )
    .orderBy("region")
)

display(region_summary)


### Most expensive product in each category

A window function keeps the product name together with the maximum price instead of returning only the maximum numeric value.


In [ ]:
price_rank = Window.partitionBy("category").orderBy(F.col("price").desc())

most_expensive_by_category = (
    sales.withColumn("price_rank", F.row_number().over(price_rank))
    .filter(F.col("price_rank") == 1)
    .select("category", "product", "price")
    .orderBy("category")
)

display(most_expensive_by_category)


## 5. Join sales with region metadata


In [ ]:
enriched_sales = (
    sales.join(regions, on="region", how="inner")
    .select("sale_id", "product", "price", "quantity", "region", "territory", "manager")
)

display(enriched_sales)


In [ ]:
territory_summary = (
    enriched_sales.groupBy("territory", "manager")
    .agg(
        F.round(F.sum(F.col("price") * F.col("quantity")), 2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

display(territory_summary)


## 6. SQL equivalent: top products by revenue


In [ ]:
sales.createOrReplaceTempView("sales")


In [ ]:
%sql
SELECT
    product,
    ROUND(SUM(price * quantity), 2) AS total_revenue
FROM sales
GROUP BY product
ORDER BY total_revenue DESC
LIMIT 3


## Key takeaways

- PySpark expressions can be composed to build readable transformation pipelines.
- Aggregations become more useful when business metrics are explicitly named.
- Joins connect transactional data with descriptive reference data.
- Window functions solve ranking problems while preserving row context.
- Spark SQL and the DataFrame API can operate on the same underlying data.
